# MH-03. X (baseline <= index): raw_value + observed_mask + days_since

For each feature: raw_value = observation closest to and on-or-before index (latest qualifying
day; ties broken by last source datetime/id, deterministic). observed_mask = 1 if any value
<= index. days_since = index - obs date (NaN if unobserved). No imputation here (raw superset).

`days_since` is computed and delivered for audit but is NOT fed to the PCA reducer in MH-04.

Blocks tagged per feature: Core / Medications & screens / Substance & misc / Observation-intensity.
This skeleton wires measurement + a couple of blocks; extend concept sets inline where marked.


## 1. Config

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import duckdb

DATA_PATH = '/home/jupyter/2836994-data2'
OUTPUT_DIR = Path('output/mh')
con = duckdb.connect()

spine = pd.read_parquet(OUTPUT_DIR / 'spine.parquet')[['person_id','index_date']]
spine['index_date'] = pd.to_datetime(spine['index_date'])
con.register('spine', spine)
persons = spine['person_id'].tolist()

def gp(table):
    return f"read_parquet('{DATA_PATH}/{table}/*.parquet')"

# ---- Concept sets to curate (fill with real concept_ids / source patterns) ----
# Keep these explicit so the feature manifest is auditable.
MEASUREMENT_CONCEPTS = {
    # 'BMI': [3038553], 'HbA1c': [...], 'LDL': [...], ...  <- curate on Kamino
}
print('persons:', len(persons), '| measurement feature groups:', len(MEASUREMENT_CONCEPTS))


## 2. As-of-index picker (generic, on-or-before index)

In [ ]:
def as_of_index(events, value_col):
    """events: person_id, feat, event_date, value_col, src_id. Returns per (person,feat)
    the latest on-or-before-index value with observed_mask and days_since."""
    e = events.merge(spine, on='person_id', how='inner')
    e['event_date'] = pd.to_datetime(e['event_date'])
    e = e[e['event_date'] <= e['index_date']].copy()
    # latest qualifying day, then last src_id for determinism
    e = e.sort_values(['person_id','feat','event_date','src_id'])
    picked = e.groupby(['person_id','feat'], as_index=False).last()
    picked['days_since'] = (picked['index_date'] - picked['event_date']).dt.days
    picked = picked.rename(columns={value_col:'raw_value'})
    return picked[['person_id','feat','raw_value','days_since']]

def to_wide(picked, block):
    """Wide raw_value + observed_mask + days_since, reindexed to full cohort."""
    raw = picked.pivot(index='person_id', columns='feat', values='raw_value').reindex(persons)
    dsi = picked.pivot(index='person_id', columns='feat', values='days_since').reindex(persons)
    mask = raw.notna().astype('int8')
    raw.columns = [f'{block}__{c}__raw' for c in raw.columns]
    mask.columns = [f'{block}__{c}__mask' for c in mask.columns]
    dsi.columns = [f'{block}__{c}__days_since' for c in dsi.columns]
    return pd.concat([raw, mask, dsi], axis=1)


## 3. Measurement block (labs / vitals) -> raw/mask/days_since

In [ ]:
blocks = []

if MEASUREMENT_CONCEPTS:
    flat = [(name, cid) for name, ids in MEASUREMENT_CONCEPTS.items() for cid in ids]
    cmap = pd.DataFrame(flat, columns=['feat','measurement_concept_id'])
    con.register('cmap', cmap)
    meas = con.sql(f'''
    SELECT m.person_id, cmap.feat,
           m.measurement_date AS event_date,
           m.value_as_number  AS value,
           m.measurement_id   AS src_id
    FROM {gp('measurement')} m
    JOIN cmap ON m.measurement_concept_id = cmap.measurement_concept_id
    WHERE m.value_as_number IS NOT NULL
    ''').to_df()
    picked = as_of_index(meas, 'value')
    blocks.append(to_wide(picked, 'Core'))
    print('Core measurement block:', blocks[-1].shape)
else:
    print('MEASUREMENT_CONCEPTS empty -> skipping measurement block (curate concept ids)')

# TODO extend: drug_exposure (psychotropic classes), observation (PHQ/GAD/C-SSRS, substance),
# person/payer demographics+SES, visit_occurrence (obs-intensity). Same as_of_index pattern.


## 4. Observation-intensity block (counts, kept raw outside reducer)

In [ ]:
obs_int = con.sql(f'''
WITH v AS (
    SELECT person_id, visit_start_date::DATE AS d
    FROM {gp('visit_occurrence')}
)
SELECT s.person_id,
       COUNT(v.d)                         AS prior_encounters,
       COUNT(DISTINCT v.d)                AS distinct_visit_days
FROM spine s
LEFT JOIN v ON v.person_id = s.person_id AND v.d <= s.index_date
GROUP BY s.person_id
''').to_df().set_index('person_id').reindex(persons)
obs_int.columns = [f'ObsIntensity__{c}__raw' for c in obs_int.columns]
blocks.append(obs_int.fillna(0))
print('obs-intensity block:', obs_int.shape)


## 5. Assemble X (row-aligned, ascending by person_id)

In [ ]:
X = pd.concat(blocks, axis=1) if blocks else pd.DataFrame(index=persons)
X.index.name = 'person_id'
X = X.sort_index().reset_index()
assert X['person_id'].tolist() == sorted(persons)
X.to_parquet(OUTPUT_DIR / 'X_raw.parquet', index=False)
print('X_raw:', X.shape)
print('columns by suffix:',
      {s: sum(c.endswith(s) for c in X.columns) for s in ['__raw','__mask','__days_since']})
